# Baseline 1a: Keyword Search Using TF-IDF Vector Space Model & Cosine Similarity

## 1. Advanced Mathematical Theory (For Q&A / Defense)

The Vector Space Model represents text documents as numerical vectors in a multi-dimensional space, where each dimension corresponds to a unique term in the vocabulary.

### 1.1 Term Frequency - TF
Measures the local importance of a term $t$ in a document $d$. If the term appears more frequently, its weight is larger.
$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$
*(Where $f_{t,d}$ is the raw count of term $t$ in document $d$, and the denominator is the total number of terms in document $d$).*

### 1.2 Inverse Document Frequency - IDF
Measures the informative power of a term $t$ across the entire corpus $D$. Terms that appear in too many documents (e.g., 'the', 'is') carry little discriminative power $\to$ low IDF. Rare terms $\to$ high IDF.
$$\text{IDF}(t, D) = \log \frac{|D|}{1 + |\{d \in D : t \in d\}|}$$
*(The denominator is added with 1 to avoid division by zero when the term does not appear in any document - Smooth IDF).*

### 1.3 TF-IDF Weighting
Combines local importance (TF) and global frequency (IDF):
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

### 1.4 Cosine Similarity
Measures the angle between the two vectors representing the query $q$ and document $d$. Cosine similarity ignores document length, focusing only on the vector directions (topical correlation).
$$\cos(\theta) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \|\vec{d}\|} = \frac{\sum_{i} q_i d_i}{\sqrt{\sum_i q_i^2} \sqrt{\sum_i d_i^2}}$$
*(The numerator is the dot product, and the denominator is the product of the L2-Norm lengths of the two vectors).*

## 2. Initialize Toy Corpus (Sample Data)
Build 5 sample sentences containing financial terms about revenue (revenue, sales) and spending (capex, purchases) to demonstrate the **Lexical Gap** weakness.

In [1]:
import numpy as np
import pandas as pd
import re

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

for k, v in corpus.items():
    print(f"{k}: {v}")

doc1: Apple Net sales in fiscal year 2023 were $383,285 million.
doc2: Apple payments for acquisition of property plant and equipment were $10,959 million.
doc3: Amazon net sales were $574,785 million in fiscal year 2023.
doc4: Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.
doc5: NVIDIA net income was $29,760 million in fiscal year 2024.


## 3. Manual TF-IDF Indexing (Step-by-Step Matrix Construction)
We will write Python code from scratch to build the vocabulary, calculate TF, IDF, and the raw TF-IDF matrix to print the detailed intermediate calculation steps.

In [2]:
# 3.1 Tokenization & Lowercasing
def tokenize(text):
    text = text.lower()
    # Remove special characters, keep only alphanumeric characters and dollar signs
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
print("[Log] Tokenized Documents:")
for k, v in tokenized_docs.items():
    print(f"  - {k}: {v}")

# 3.2 Build the Vocabulary
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))
print(f"\n[Log] Vocabulary Size: {len(vocabulary)} terms")
print(f"Vocabulary: {vocabulary}")

[Log] Tokenized Documents:
  - doc1: ['apple', 'net', 'sales', 'in', 'fiscal', 'year', '2023', 'were', '$383285', 'million']
  - doc2: ['apple', 'payments', 'for', 'acquisition', 'of', 'property', 'plant', 'and', 'equipment', 'were', '$10959', 'million']
  - doc3: ['amazon', 'net', 'sales', 'were', '$574785', 'million', 'in', 'fiscal', 'year', '2023']
  - doc4: ['amazon', 'purchases', 'of', 'property', 'and', 'equipment', 'in', 'fiscal', 'year', '2023', 'were', '$52729', 'million']
  - doc5: ['nvidia', 'net', 'income', 'was', '$29760', 'million', 'in', 'fiscal', 'year', '2024']

[Log] Vocabulary Size: 28 terms
Vocabulary: ['$10959', '$29760', '$383285', '$52729', '$574785', '2023', '2024', 'acquisition', 'amazon', 'and', 'apple', 'equipment', 'fiscal', 'for', 'in', 'income', 'million', 'net', 'nvidia', 'of', 'payments', 'plant', 'property', 'purchases', 'sales', 'was', 'were', 'year']


### 3.3 Compute Term Frequency Matrix - TF Matrix
Print the entire TF matrix as a DataFrame to examine each term in detail.

In [3]:
tf_records = []
for doc_name, tokens in tokenized_docs.items():
    total_words = len(tokens)
    doc_tf = {}
    for word in vocabulary:
        count = tokens.count(word)
        doc_tf[word] = count / total_words  # Normalized frequency
    tf_records.append(doc_tf)

df_tf = pd.DataFrame(tf_records, index=corpus.keys())
print("[SUCCESS] TF matrix calculation complete!")

# Display the entire TF matrix (without truncating any columns)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n=== COMPLETE TF MATRIX (DOCUMENT-TERM MATRIX) ===")
print(df_tf.to_string())
df_tf

[SUCCESS] TF matrix calculation complete!

=== COMPLETE TF MATRIX (DOCUMENT-TERM MATRIX) ===
        $10959  $29760  $383285    $52729  $574785      2023  2024  acquisition    amazon       and     apple  equipment    fiscal       for        in  income   million  net  nvidia        of  payments     plant  property  purchases  sales  was      were      year
doc1  0.000000     0.0      0.1  0.000000      0.0  0.100000   0.0     0.000000  0.000000  0.000000  0.100000   0.000000  0.100000  0.000000  0.100000     0.0  0.100000  0.1     0.0  0.000000  0.000000  0.000000  0.000000   0.000000    0.1  0.0  0.100000  0.100000
doc2  0.083333     0.0      0.0  0.000000      0.0  0.000000   0.0     0.083333  0.000000  0.083333  0.083333   0.083333  0.000000  0.083333  0.000000     0.0  0.083333  0.0     0.0  0.083333  0.083333  0.083333  0.083333   0.000000    0.0  0.0  0.083333  0.000000
doc3  0.000000     0.0      0.0  0.000000      0.1  0.100000   0.0     0.000000  0.100000  0.000000  0.000000   

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.0,0.1,0.000000,0.0,0.100000,0.0,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.000000,0.100000,0.0,0.100000,0.1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.1,0.0,0.100000,0.100000
doc2,0.083333,0.0,0.0,0.000000,0.0,0.000000,0.0,0.083333,0.000000,0.083333,0.083333,0.083333,0.000000,0.083333,0.000000,0.0,0.083333,0.0,0.0,0.083333,0.083333,0.083333,0.083333,0.000000,0.0,0.0,0.083333,0.000000
doc3,0.000000,0.0,0.0,0.000000,0.1,0.100000,0.0,0.000000,0.100000,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.0,0.100000,0.1,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.1,0.0,0.100000,0.100000
doc4,0.000000,0.0,0.0,0.076923,0.0,0.076923,0.0,0.000000,0.076923,0.076923,0.000000,0.076923,0.076923,0.000000,0.076923,0.0,0.076923,0.0,0.0,0.076923,0.000000,0.000000,0.076923,0.076923,0.0,0.0,0.076923,0.076923
doc5,0.000000,0.1,0.0,0.000000,0.0,0.000000,0.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.100000,0.000000,0.100000,0.1,0.100000,0.1,0.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.1,0.000000,0.100000


### 3.4 Compute IDF Value for Each Term
Print all IDF values for the vocabulary in the corpus.

In [4]:
num_docs = len(corpus)
idf_dict = {}
for word in vocabulary:
    # Count the number of documents containing the word
    containing_docs = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    # Calculate IDF with the smooth formula
    idf = np.log(num_docs / (1 + containing_docs)) # Smooth IDF
    idf_dict[word] = idf

df_idf = pd.Series(idf_dict).sort_values(ascending=False)
print("[SUCCESS] IDF calculation completed for all vocabulary!")
print("\n=== COMPLETE IDF VALUES OF VOCABULARY (Descending) ===")
df_idf_df = pd.DataFrame({'Word': df_idf.index, 'IDF': df_idf.values})
print(df_idf_df.to_string(index=False))
df_idf_df

[SUCCESS] IDF calculation completed for all vocabulary!

=== COMPLETE IDF VALUES OF VOCABULARY (Descending) ===
       Word       IDF
     $10959  0.916291
     $29760  0.916291
    $383285  0.916291
     $52729  0.916291
    $574785  0.916291
       2024  0.916291
acquisition  0.916291
        was  0.916291
   payments  0.916291
     nvidia  0.916291
     income  0.916291
        for  0.916291
      plant  0.916291
  purchases  0.916291
   property  0.510826
      apple  0.510826
     amazon  0.510826
  equipment  0.510826
        and  0.510826
         of  0.510826
      sales  0.510826
       2023  0.223144
        net  0.223144
     fiscal  0.000000
       were  0.000000
         in  0.000000
       year  0.000000
    million -0.182322


,Word,IDF
0,$10959,0.916291
1,$29760,0.916291
2,$383285,0.916291
3,$52729,0.916291
4,$574785,0.916291
5,2024,0.916291
6,acquisition,0.916291
7,was,0.916291
8,payments,0.916291
9,nvidia,0.916291


### 3.5 Compute the Final Weight Matrix - TF-IDF Matrix
Multiply the TF matrix by the corresponding IDF vector.

In [5]:
tfidf_records = []
for doc_name, tokens in tokenized_docs.items():
    doc_tfidf = {}
    total_words = len(tokens)
    for word in vocabulary:
        tf = tokens.count(word) / total_words
        idf = idf_dict[word]
        doc_tfidf[word] = tf * idf
    tfidf_records.append(doc_tfidf)

df_tfidf = pd.DataFrame(tfidf_records, index=corpus.keys())
print("[SUCCESS] TF-IDF matrix built successfully!")

print("\n=== COMPLETE TF-IDF MATRIX ===")
print(df_tfidf.to_string())
df_tfidf

[SUCCESS] TF-IDF matrix built successfully!

=== COMPLETE TF-IDF MATRIX ===
        $10959    $29760   $383285    $52729   $574785      2023      2024  acquisition    amazon       and     apple  equipment  fiscal       for   in    income   million       net    nvidia        of  payments     plant  property  purchases     sales       was  were  year
doc1  0.000000  0.000000  0.091629  0.000000  0.000000  0.022314  0.000000     0.000000  0.000000  0.000000  0.051083   0.000000     0.0  0.000000  0.0  0.000000 -0.018232  0.022314  0.000000  0.000000  0.000000  0.000000  0.000000   0.000000  0.051083  0.000000   0.0   0.0
doc2  0.076358  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000     0.076358  0.000000  0.042569  0.042569   0.042569     0.0  0.076358  0.0  0.000000 -0.015193  0.000000  0.000000  0.042569  0.076358  0.076358  0.042569   0.000000  0.000000  0.000000   0.0   0.0
doc3  0.000000  0.000000  0.000000  0.000000  0.091629  0.022314  0.000000     0.000000  0.051083  

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.000000,0.091629,0.000000,0.000000,0.022314,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.000000,0.0,0.000000,-0.018232,0.022314,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.0
doc2,0.076358,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.076358,0.000000,0.042569,0.042569,0.042569,0.0,0.076358,0.0,0.000000,-0.015193,0.000000,0.000000,0.042569,0.076358,0.076358,0.042569,0.000000,0.000000,0.000000,0.0,0.0
doc3,0.000000,0.000000,0.000000,0.000000,0.091629,0.022314,0.000000,0.000000,0.051083,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,-0.018232,0.022314,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.051083,0.000000,0.0,0.0
doc4,0.000000,0.000000,0.000000,0.070484,0.000000,0.017165,0.000000,0.000000,0.039294,0.039294,0.000000,0.039294,0.0,0.000000,0.0,0.000000,-0.014025,0.000000,0.000000,0.039294,0.000000,0.000000,0.039294,0.070484,0.000000,0.000000,0.0,0.0
doc5,0.000000,0.091629,0.000000,0.000000,0.000000,0.000000,0.091629,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.091629,-0.018232,0.022314,0.091629,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.091629,0.0,0.0


## 4. Execute Retrieval Process
Receive query $\to$ Convert to TF-IDF vector $\to$ Compute Cosine Similarity against all documents in the TF-IDF matrix.

In [6]:
def search_tfidf(query):
    print(f"\n=== USER QUERY: '{query}' ===")
    query_tokens = tokenize(query)
    print(f"[Step 1] Tokenized Query: {query_tokens}")
    
    # Construct TF-IDF Vector for the Query
    query_vector = np.zeros(len(vocabulary))
    total_q_words = len(query_tokens) if len(query_tokens) > 0 else 1
    
    print("\n[Log] Detailed TF-IDF calculation for each word in the Query:")
    for word in query_tokens:
        if word in vocabulary:
            word_idx = vocabulary.index(word)
            tf = query_tokens.count(word) / total_q_words
            idf = idf_dict[word]
            query_vector[word_idx] = tf * idf
            print(f"  - Word '{word}' (Index {word_idx}): tf = {query_tokens.count(word)}/{total_q_words} = {tf:.4f}, idf = {idf:.4f} -> tf-idf = {tf*idf:.4f}")
        else:
            print(f"  - Word '{word}': Not in vocabulary (Out of Vocabulary - OOV)")
            
    # Print the complete Query Vector
    print(f"\nQuery TF-IDF Vector (length of {len(vocabulary)} dimensions):\n{query_vector}")
    
    # Calculate Cosine Similarity with each document
    q_norm = np.linalg.norm(query_vector)
    print(f"\n[Step 2] Length of Query Vector (L2 Norm): ||q|| = {q_norm:.4f}")
    
    scores = {}
    for doc_name in corpus.keys():
        doc_vector = df_tfidf.loc[doc_name].values
        doc_norm = np.linalg.norm(doc_vector)
        
        # Calculate Dot Product and print detailed multiplication for each dimension
        dot_product = 0.0
        multiplications = []
        for idx, word in enumerate(vocabulary):
            q_val = query_vector[idx]
            d_val = doc_vector[idx]
            if q_val > 0 and d_val > 0:
                prod = q_val * d_val
                dot_product += prod
                multiplications.append(f"('{word}': q_tfidf={q_val:.4f} * d_tfidf={d_val:.4f} = {prod:.4f})")
        
        # Calculate cosine similarity
        if q_norm > 0 and doc_norm > 0:
            cosine_sim = dot_product / (q_norm * doc_norm)
        else:
            cosine_sim = 0.0
            
        scores[doc_name] = cosine_sim
        print(f"\n* Comparing with {doc_name}:")
        print(f"  - Vector of {doc_name}:\n    {doc_vector}")
        print(f"  - Vector length ||d|| = {doc_norm:.4f}")
        print(f"  - Element-wise multiplications: {', '.join(multiplications) if multiplications else 'No matching words (product = 0)'}")
        print(f"  - Dot Product = {dot_product:.4f}")
        print(f"  - Cosine Similarity = {dot_product:.4f} / ({q_norm:.4f} * {doc_norm:.4f}) = {cosine_sim:.4f}")
        
    # Sort and rank
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\n=== RETRIEVAL RANKING RESULTS ===")
    for rank, (doc_name, score) in enumerate(sorted_scores):
        print(f"  Rank {rank+1}: {doc_name} | Cosine Score: {score:.4f} | Content: {corpus[doc_name]}")
        
    return sorted_scores

## 5. Error and Weakness Analysis (Failure Analysis)

Now, we run two test queries to clearly show the limitations of TF-IDF.

In [7]:
# Test 1: Exact Keyword Match
search_tfidf("Apple Net sales in 2023")


=== USER QUERY: 'Apple Net sales in 2023' ===
[Step 1] Tokenized Query: ['apple', 'net', 'sales', 'in', '2023']

[Log] Detailed TF-IDF calculation for each word in the Query:
  - Word 'apple' (Index 10): tf = 1/5 = 0.2000, idf = 0.5108 -> tf-idf = 0.1022
  - Word 'net' (Index 17): tf = 1/5 = 0.2000, idf = 0.2231 -> tf-idf = 0.0446
  - Word 'sales' (Index 24): tf = 1/5 = 0.2000, idf = 0.5108 -> tf-idf = 0.1022
  - Word 'in' (Index 14): tf = 1/5 = 0.2000, idf = 0.0000 -> tf-idf = 0.0000
  - Word '2023' (Index 5): tf = 1/5 = 0.2000, idf = 0.2231 -> tf-idf = 0.0446

Query TF-IDF Vector (length of 28 dimensions):
[0.         0.         0.         0.         0.         0.04462871
 0.         0.         0.         0.         0.10216512 0.
 0.         0.         0.         0.         0.         0.04462871
 0.         0.         0.         0.         0.         0.
 0.10216512 0.         0.         0.        ]

[Step 2] Length of Query Vector (L2 Norm): ||q|| = 0.1577

* Comparing with doc1:
  

[('doc1', np.float64(0.644898785818549)),
 ('doc3', np.float64(0.37411944104748696)),
 ('doc2', np.float64(0.14068266639609808)),
 ('doc4', np.float64(0.03606669664087338)),
 ('doc5', np.float64(0.030527167536609653))]

In [8]:
# Test 2: Financial Synonym (Lexical Gap)
# Query uses 'capital expenditures'
# But Amazon's document (doc4) uses 'purchases of property and equipment'
search_tfidf("Amazon capital expenditures 2023")


=== USER QUERY: 'Amazon capital expenditures 2023' ===
[Step 1] Tokenized Query: ['amazon', 'capital', 'expenditures', '2023']

[Log] Detailed TF-IDF calculation for each word in the Query:
  - Word 'amazon' (Index 8): tf = 1/4 = 0.2500, idf = 0.5108 -> tf-idf = 0.1277
  - Word 'capital': Not in vocabulary (Out of Vocabulary - OOV)
  - Word 'expenditures': Not in vocabulary (Out of Vocabulary - OOV)
  - Word '2023' (Index 5): tf = 1/4 = 0.2500, idf = 0.2231 -> tf-idf = 0.0558

Query TF-IDF Vector (length of 28 dimensions):
[0.         0.         0.         0.         0.         0.05578589
 0.         0.         0.12770641 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ]

[Step 2] Length of Query Vector (L2 Norm): ||q|| = 0.1394

* Comparing with doc1:
  - Vector of doc1:
    [ 0.          0.          0.09162907  0.          0.          0.02231436


[('doc3', np.float64(0.45601230463126696)),
 ('doc4', np.float64(0.31830544093258173)),
 ('doc1', np.float64(0.07307248284553076)),
 ('doc2', np.float64(0.0)),
 ('doc5', np.float64(0.0))]

### Observations on Test 2 Failure:
- The correct document is **doc4** (`Amazon purchases of property and equipment...`).
- Ranking result: **doc4 receives a score of 0.0000** (tied for last place with doc2, doc5).
- **Root Cause:** The main keywords `capital` and `expenditures` do not appear anywhere in `doc4`. TF-IDF is incapable of capturing semantic synonyms, leading to complete retrieval failure.